In [1]:
import os 
import json
import numpy as np

from uuid import uuid5, NAMESPACE_URL
from dotenv import load_dotenv
from pathlib import Path
from qdrant_client import QdrantClient, models

In [2]:
#ROOT = Path(__file__).resolve().parent.parent
load_dotenv()
hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN is not set")

In [3]:
MODEL = "gemma4:e2b"
DB_NAME = "vector_db"
#BASE_URL = os.environ("BASE_URL")
MAX_CHARS=1200

## Spliting Documents file into list

In [4]:

DATA_PATH = Path("../data/processed/documents.jsonl")

documents = []

with DATA_PATH.open("r", encoding = "utf-8") as f:
    for line in f :
        if line.strip():
            documents.append(json.loads(line))

print("Number of documents:", len(documents))
print(documents[0].keys())
documents[0]["content"]

Number of documents: 66192
dict_keys(['document_id', 'document_type', 'source_id', 'source_name', 'message_date', 'content', 'message_ids', 'media_paths'])


'. \n🟦🟦 قابل توجه دانشجویان گرامی 🟦🟦\n\n🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩🟩\n\nکلاس\u200c اقتصاد کلان استاد کامران رحیمی روز چهارشنبه ۱۱ مهر ماه برگزار نخواهد شد. \n\nکلاس جبرانی توسط استاد اطلاع\u200cرسانی خواهد شد.\n\n🟥🟥🟥🟥🟥🟥🟥🟥🟥🟥🟥🟥🟥\n.'

In [5]:
current = ''
id=0
for i,doc in enumerate( documents):
    if len(doc['content']) >len(current) :
        current = doc["content"]
        id = i
print(len(current)) 
print(id)       

13803
45620


In [6]:
current

'[message 123305] یه کانال شمسی نیوزمون نشه؟\n[message 123306] تگ\n[message 123308] پین\n[message 123310] • پیام مورد نظر با موفقیت سنجاق شد!\n[message 123311] [A_h_shirmohammadi](tg://user?id=602026979) ⊹ [sally_ix](tg://user?id=8078076927) ⊹ [indianabol](tg://user?id=266316365) ⊹ [Alireza_Hoseinifar](tg://user?id=6343869946) ⊹ [Yunes_fff](tg://user?id=7329382783) ⊹ [Shaay69](tg://user?id=879572733)\n[message 123312] [M_amin_114](tg://user?id=6381398870) ⊹ [Yasin40Amirani](tg://user?id=5815216365) ⊹ [P0XER](tg://user?id=1904631905) ⊹ [Kiarash](tg://user?id=5588889443) ⊹ [Hosein333128](tg://user?id=1822732163) ⊹ [Ardavan9090](tg://user?id=1635050713)\n[message 123313] [Khodemehrshadd](tg://user?id=1515784250) ⊹ [Gorillla_6](tg://user?id=7441798192) ⊹ [Staff_Ashkan](tg://user?id=1779987519) ⊹ [amirDKS](tg://user?id=1494449208) ⊹ [Amiroouu](tg://user?id=6396955246) ⊹ [Sin4oheil](tg://user?id=541855127)\n[message 123314] [Computer_MF_2005](tg://user?id=1449481794) ⊹ [keyumarsss](tg://user

## Split Documents to chunks

In [7]:
def split_into_chunks(text, max_chars=1200):
    paragraphs = [
        paragraph.strip()
        for paragraph in text.split("\n\n")
        if paragraph.strip()
    ]

    chunks = []
    current = ""

    for paragraph in paragraphs:

        # اگر خود paragraph بزرگ‌تر از حد مجاز است
        if len(paragraph) > max_chars:
            if current:
                chunks.append(current)
                current = ""

            start = 0
            while start < len(paragraph):
                chunks.append(paragraph[start:start + max_chars])
                start += max_chars

            continue

        candidate = (
            paragraph
            if not current
            else current + "\n\n" + paragraph
        )

        if len(candidate) <= max_chars:
            current = candidate
        else:
            chunks.append(current)
            current = paragraph

    if current:
        chunks.append(current)

    return chunks

In [8]:
for doc in documents:
    chunks = split_into_chunks(doc["content"], MAX_CHARS)
    doc["chunks"] = chunks

## Turning Chunks into embedding

In [9]:
import torch
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "Alibaba-NLP/gte-multilingual-base",
    trust_remote_code=True
)

# Workaround for the model's non-persistent position_ids buffer
auto_model = embedding_model[0].auto_model
model_device = next(auto_model.parameters()).device
auto_model.embeddings.position_ids = torch.arange(
    auto_model.config.max_position_embeddings,
    device=model_device,
    dtype=torch.long,
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

NewModel LOAD REPORT from: Alibaba-NLP/gte-multilingual-base
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:


records = []

for doc_index, doc in enumerate(documents):
    metadata = {}
    for key, value in doc.items():
        if key not in ("chunks", "content"):
            metadata[key]=value
        
    

    for chunk_index, text in enumerate(doc["chunks"]):
        records.append({
            "id": str(uuid5(
                NAMESPACE_URL,
                f"campus-rag:{doc_index}:{chunk_index}"
            )),
            "text": text,
            "metadata": metadata
        })

texts = [record["text"] for record in records]

vectors = embedding_model.encode(
    texts,
    batch_size=16,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

for record, vector in zip(records, vectors):
    record["vector"] = vector.astype(np.float32).tolist()

print(len(records))
print(len(records[0]["vector"]))

Batches:   0%|          | 0/4221 [00:00<?, ?it/s]

67527
768


In [ ]:

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

db_path = project_root / "data" / "vector_db"

client = QdrantClient(path=str(db_path))

collection_name = "campus_chunks"

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=768,
            distance=models.Distance.COSINE
        )
    )

Qdrant آماده است


C:\Users\LOQ\AppData\Local\Temp\ipykernel_15388\4217378932.py:10: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <campus_chunks> contains 67527 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = QdrantClient(path=str(db_path))


In [12]:
batch_size = 64

for start in range(0, len(records), batch_size):
    batch = records[start:start + batch_size]

    points = [
        models.PointStruct(
            id=record["id"],
            vector=record["vector"],
            payload={
                "text": record["text"],
                "metadata": record["metadata"]
            }
        )
        for record in batch
    ]

    client.upsert(
        collection_name=collection_name,
        points=points
    )

print("تعداد رکوردها:", len(records))
print(
    "تعداد pointهای Qdrant:",
    client.count(collection_name).count
)

C:\Users\LOQ\AppData\Local\Temp\ipykernel_15388\2746725491.py:18: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20032 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(


تعداد رکوردها: 67527
تعداد pointهای Qdrant: 67527


In [13]:
client.create_payload_index(
    collection_name=collection_name,
    field_name="text",
    field_schema=models.TextIndexParams(
        type=models.TextIndexType.TEXT,
        tokenizer=models.TokenizerType.MULTILINGUAL,
        min_token_len=2,
        max_token_len=30,
        lowercase=True,
    )
)

print("Text index ساخته شد")

Text index ساخته شد


C:\Users\LOQ\AppData\Local\Temp\ipykernel_15388\3556706890.py:1: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(


In [41]:
import re
import numpy as np
from rank_bm25 import BM25Okapi

def tokenize_fa(text):
    text = str(text).lower()
    text = text.replace("ي", "ی").replace("ك", "ک")
    text = text.replace("\u200c", " ")
    return re.findall(r"\w+", text)

tokenized_records = [
    tokenize_fa(record["text"])
    for record in records
]

bm25 = BM25Okapi(tokenized_records)

print("BM25 آماده شد:", len(tokenized_records))

BM25 آماده شد: 67527


In [45]:
def hybrid_search(query, top_k=10, candidate_k=50):
    # جست‌وجوی متنی با BM25
    query_tokens = tokenize_fa(query)
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_indices = [
        index
        for index in np.argsort(bm25_scores)[::-1]
        if bm25_scores[index] > 0
    ][:candidate_k]
    bm25_ids = [str(records[index]["id"]) for index in bm25_indices]

    # جست‌وجوی معنایی با embedding در Qdrant
    query_vector = embedding_model.encode(
        query,
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32").tolist()

    dense_results = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=candidate_k,
        with_payload=True
    ).points
    dense_ids = [str(result.id) for result in dense_results]

    # ترکیب رتبه‌های دو جست‌وجو با RRF
    rrf_scores = {}
    for ranked_ids in (bm25_ids, dense_ids):
        for rank, record_id in enumerate(ranked_ids, start=1):
            rrf_scores[record_id] = rrf_scores.get(record_id, 0)
            rrf_scores[record_id] += 1 / (60 + rank)

    ranked_ids = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )[:top_k]

    records_by_id = {str(record["id"]): record for record in records}
    return [
        {"record": records_by_id[record_id], "score": rrf_scores[record_id]}
        for record_id in ranked_ids
        if record_id in records_by_id
    ]

query = "استاد شیروانی کیه؟"
results = hybrid_search(query, top_k=50)

for result in results:
    print("RRF Score:", result["score"])
    print(result["record"]["text"])
    print("-" * 50)

RRF Score: 0.01639344262295082
[message 7373] دوستان هر کسی کتاب اندیشه اسلامی ۱(استاد رهبرخواه) و زبان انگلیسی general English(استاد شیروانی) خواست دارم بیاد پیوی
[message 7459] سلام نویسنده اندیشه ۱ کیه؟
[message 7482] جعفر سبحانی
--------------------------------------------------
RRF Score: 0.01639344262295082
[message 44093] خارکسه بازیا چیه وقتی گفتی از ۱ و ۲ میاد، چرا رفتی از ۱۰ آوردی آخه
--------------------------------------------------
RRF Score: 0.016129032258064516
[message 66528] فروش کتاب زبان عمومی استاد شیروانی
--------------------------------------------------
RRF Score: 0.016129032258064516
[message 23812] @Life_is_passing
اندیشه حسن پور داشتی؟
یه توضیح ریز میدی ببینم اوکیه یا حذف کنم
--------------------------------------------------
RRF Score: 0.015873015873015872
[message 68521] فروش کتاب زبان جدید استاد شیروانی
--------------------------------------------------
RRF Score: 0.015873015873015872
ولمون کن دا سکس چیه
--------------------------------------------------
RR